In [176]:
from openpyxl import load_workbook
from pandas import ExcelWriter
import numpy as np
import pandas as pd
import requests

In [177]:
p = 'https://www.ons.gov.uk/file?uri=%2fbusinessindustryandtrade%2fretailindustry%2fdatasets%2fretailsalesindexinternetsales%2fcurrent/internetsalesreferencetables.xlsx'
g = 'https://www.ons.gov.uk/file?uri=%2fbusinessindustryandtrade%2fretailindustry%2fdatasets%2fpoundsdatatotalretailsales%2fcurrent/poundsdataaccessible.xlsx'

with open(f"ons_xlsx_total.xlsx", "wb") as f:
    f.write(requests.get(g).content)

with open(f"ons_xlsx_online.xlsx", "wb") as f:
    f.write(requests.get(p).content)

In [178]:
def format_ons_xlsx_online():
    wb = load_workbook('ons_xlsx_online.xlsx')
    ws = wb['IntValSA'].values
    x = []
    df = pd.DataFrame(ws).iloc[3:].reset_index(drop=True)
    header = df.iloc[0]
    df = df[1:]
    df.columns = header
    df = df.dropna(axis='columns').iloc[2:].reset_index(drop=True)
    df['Time Period'] = pd.to_datetime(df['Time Period'], format='%d%b%Y:%H:%M:%S.%f')
    df = df.set_index('Time Period')
    df = df * (52/12)
    df.columns = (df.columns.str.split('[')).str[0]
    df.columns = map(str.lower, df.columns)
    df.columns = df.columns.str.strip()
    df.to_excel('ons_xlsx_online-edited.xlsx')
    return pd.DataFrame(df)

In [193]:

def format_ons_xlsx_total():
    wb = load_workbook('ons_xlsx_total.xlsx')
    ws = wb['ValSAT'].values
    x = []
    df = pd.DataFrame(ws).iloc[4:].reset_index(drop=True)
    header = df.iloc[0]
    df = df[171:]
    df.columns = header
    df = df.drop(['Automotive fuel','All retailing including automotive fuel','Month as a % of Total','Month as a % of Total excluding automotive fuel', 'Total Annual Sales for All Retailing including automotive fuel','Total Annual Sales for All Retailing excluding automotive fuel'], axis=1)
    df = df.rename(columns={df.columns[0]: 'Time Period'}).reset_index(drop=True)
    df = df.rename(columns={df.columns[3]: 'Total of predominantly non-food stores'})
    df['Time Period'] = pd.to_datetime(df['Time Period'])
    df = df.set_index('Time Period')
    df = df.dropna()
    df = df / 1000
    df.columns = map(str.lower, df.columns)
    df.columns = df.columns.str.strip()
    df.to_excel('ons_xlsx_total-edited.xlsx')
    return pd.DataFrame(df)

In [208]:
x = format_ons_xlsx_total() - format_ons_xlsx_online()

y = format_ons_xlsx_total()
y['yoy % all retailing excluding automotive fuel'] = y['all retailing excluding automotive fuel'].pct_change(periods=12) * 100
y = y[['yoy % all retailing excluding automotive fuel','predominantly food stores','total of predominantly non-food stores','non-store retailing']].dropna()

with pd.ExcelWriter('output.xlsx') as writer:  
    format_ons_xlsx_total().to_excel(writer, sheet_name='TotalValSAT')
    format_ons_xlsx_online().to_excel(writer, sheet_name='IntValSAT')
    x.to_excel(writer, sheet_name='OfflineValSAT')
    y.to_excel(writer, sheet_name='Flourish')

# What format does Flourish require?

## The charts
**Online and offline**
 - Total (YoY)	Food	Non-food	Non-store
 - t.csv, r.csv

**Online vs offline**
 - Online YoY%	Offline YoY%	Online Sales	Offline Sales
 - q.csv

**YoY % by Category**
 - Total (excl. Fuel)	Food	Non-food	Non-store
 - aggregate_df_total.csv

,yoy % all retailing excluding automotive fuel,predominantly food stores,total of predominantly non-food stores,non-store retailing
Time Period,,,,
2009-01-01,-18.837151,10036.482,10498.142,1028.333
2009-02-01,-0.998653,10186.222,10224.13,972.838
2009-03-01,1.009057,12779.86,12910.747,1301.815
2009-04-01,2.360200,10226.831,10477.943,1061.841
2009-05-01,-1.692629,10270.586,10395.477,1065.319
...,...,...,...,...
2021-09-01,-0.067998,16935.059,17110.136,6337.082
2021-10-01,0.478946,13698.829,14274.773,5056.208
2021-11-01,5.908615,13747.921,14464.685,5187.203


In [ ]:
r = retail_sales_internet.drop(columns_to_drop, axis=1).rename(columns=rename_column)
r['total excl. fuel'] = r['total excl. fuel'].pct_change(periods=12)
r.drop(r[r.index < '2009-01-01'].index).to_csv('r.csv')

t = retail_sales_offline.drop(columns_to_drop, axis=1).rename(columns=rename_column)
t['total excl. fuel'] = t['total excl. fuel'].pct_change(periods=12)
t.drop(t[t.index < '2009-01-01'].index).to_csv('t.csv')

q = retail_sales_internet.drop(columns_to_drop, axis=1).rename(columns=rename_column)
q = q.iloc[:, :1].rename(columns={'total excl. fuel':'total_online'})
q['total_offline'] = retail_sales_offline.drop(columns_to_drop, axis=1).rename(columns=rename_column).iloc[:, :1]
q['yoy_online'] = q.total_online.pct_change(periods=12)
q['yoy_offline'] = q.total_offline.pct_change(periods=12)
q.drop(q[q.index < '2009-01-01'].index).to_csv('q.csv')